In [ ]:
# =======================================================
# ES3 — Content-to-form: definizione -> synset WordNet
# =======================================================

!pip install pandas numpy openpyxl nltk sentence-transformers

In [ ]:
import re
import numpy as np
import pandas as pd
from itertools import combinations
from typing import List, Dict, Tuple, Optional

import nltk
nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("punkt")
nltk.download('punkt_tab')

from nltk.corpus import wordnet as wn
from nltk.tokenize import word_tokenize

from sentence_transformers import SentenceTransformer

# -----------------------------
# 0) Caricamento e normalizzazione dataset
# -----------------------------
DATA_PATH = "dataset_definizioni_TLN_25.xlsx"
df = pd.read_excel(DATA_PATH)
df.columns = [str(c).strip() for c in df.columns]

CODES = {"CG", "CS", "AG", "AS"}

def find_category_column(df: pd.DataFrame) -> str:
    for col in df.columns:
        vals = df[col].dropna().astype(str).str.strip().unique().tolist()
        if len(vals) > 0:
            hit = sum(v in CODES for v in vals)
            if hit >= max(1, int(0.5 * len(vals))):
                return col
    raise ValueError("Non trovo la colonna categoria (CG/CS/AG/AS).")

def find_term_column(df: pd.DataFrame, cat_col: str) -> str:
    for cand in ["Termine", "termine", "TERMINE"]:
        if cand in df.columns:
            return cand
    cols = list(df.columns)
    if cols[0] == cat_col and len(cols) >= 2:
        return cols[1]
    raise ValueError("Non trovo la colonna 'Termine'.")

cat_col = find_category_column(df)
term_col = find_term_column(df, cat_col)
def_cols = [c for c in df.columns if c not in {cat_col, term_col}]

def get_definitions_for_row(row: pd.Series) -> List[str]:
    defs = []
    for c in def_cols:
        v = row.get(c, None)
        if isinstance(v, str) and v.strip():
            defs.append(v.strip())
    return defs

# -----------------------------
# 1) Modello embeddings (multi-lingua)
# -----------------------------
SEM_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(SEM_MODEL)

def cosine(a: np.ndarray, b: np.ndarray) -> float:
    na = np.linalg.norm(a)
    nb = np.linalg.norm(b)
    if na == 0 or nb == 0:
        return 0.0
    return float(np.dot(a, b) / (na * nb))

# trasforma testi in vettori numerici normalizzati
def encode(texts: List[str]) -> np.ndarray:
    # normalize_embeddings=True => cosine più stabile
    return model.encode(texts, normalize_embeddings=True, show_progress_bar=False)

# -----------------------------
# 2) Genus extraction (euristico, IT)
# -----------------------------
ARTICLES = {"un", "uno", "una", "l'", "il", "lo", "la", "i", "gli", "le"}

def normalize_token(t: str) -> str:
    t = t.lower().strip()
    t = re.sub(r"[^a-zàèéìòù']", "", t)
    return t

def extract_genus(definition: str) -> Optional[str]:
    """
    Cerca pattern tipo "X è un/una Y ..." oppure "X: un/una Y..."
    Ritorna Y (prima parola contenuto dopo l'articolo).
    """
    if not definition or not isinstance(definition, str):
        return None

    text = definition.lower()

    # pattern 1: "è un/una/uno [GENUS]"
    m = re.search(r"\bè\s+(un|una|uno)\s+([a-zàèéìòù']+)", text)
    if m:
        return normalize_token(m.group(2))

    # pattern 2: "un/una/uno [GENUS]"
    m = re.search(r"^\s*(un|una|uno)\s+([a-zàèéìòù']+)", text)
    if m:
        return normalize_token(m.group(2))

    # fallback: prima parola significativa
    toks = [normalize_token(t) for t in word_tokenize(text)]
    toks = [t for t in toks if t and t not in ARTICLES]
    return toks[0] if toks else None

# -----------------------------
# 3) WordNet: genus -> candidati (iponimi)
# -----------------------------
def synsets_by_lemma_it(lemma_it: str) -> List[wn.synset]:
    lemma_it = lemma_it.lower().strip()
    out = []
    # scorre tutti i synset che hanno il lemma cercato registrato in italiano
    for syn in wn.all_synsets():
        if lemma_it in (l.lower() for l in syn.lemma_names(lang="ita")):
            out.append(syn)
    return out

def closure_hyponyms(start_synsets: List[wn.synset], depth: int = 2, max_nodes: int = 3000) -> List[wn.synset]:
    """
    Dato il genus, esplora l'albero di WordNet verso il basso (iponimi) per trovare termini più specifici.
    """
    seen = set()
    frontier = list(start_synsets)
    for s in start_synsets:
        seen.add(s.name())

    for _ in range(depth): # profondità di ricerca
        new_frontier = []
        for syn in frontier:
            for h in syn.hyponyms():
                if h.name() not in seen:
                    seen.add(h.name())
                    new_frontier.append(h)
                    if len(seen) >= max_nodes:
                        break
            if len(seen) >= max_nodes:
                break
        frontier = new_frontier
        if not frontier:
            break

    # ritorna gli oggetti synset dai nomi unici raccolti
    name2syn = {}
    for syn in start_synsets:
        name2syn[syn.name()] = syn
    for syn in wn.all_synsets():
        n = syn.name()
        if n in seen:
            name2syn[n] = syn
    return list(name2syn.values())

def candidate_synsets_from_definition(definition: str, depth: int = 2) -> Tuple[Optional[str], List[wn.synset]]:
    genus = extract_genus(definition)
    if not genus:
        return None, []

    genus_syns_it = synsets_by_lemma_it(genus)

    # fallback: se non trovo genus in italiano, provo come inglese
    if not genus_syns_it:
        genus_syns_it = wn.synsets(genus)

    if not genus_syns_it:
        return genus, []

    candidates = closure_hyponyms(genus_syns_it, depth=depth)
    return genus, candidates

# -----------------------------
# 4) Scoring e selezione del miglior synset
# -----------------------------
def score_synset(definition: str, syn: wn.synset, def_emb: np.ndarray) -> float:
    gloss = syn.definition()  # inglese
    gloss_emb = encode([gloss])[0]

    # similarità semantica tra definizione IT e glossa EN di WordNet
    sem = cosine(def_emb, gloss_emb)

    # piccolo bonus se qualche lemma italiano compare nella definizione (debole ma utile)
    def_low = definition.lower()
    lemmas_it = [l.lower().replace("_", " ") for l in syn.lemma_names(lang="ita")]
    bonus = 0.0
    for li in lemmas_it:
        if li and li in def_low:
            bonus = 0.05
            break

    return sem + bonus

def best_synsets_for_definition(definition: str, topk: int = 5, depth: int = 2) -> Tuple[Optional[str], List[Tuple[wn.synset, float]]]:
    """Estrae genus -> trova candidati -> score"""
    genus, candidates = candidate_synsets_from_definition(definition, depth=depth)
    if not candidates:
        return genus, []

    def_emb = encode([definition])[0]
    scored = []
    for syn in candidates:
        s = score_synset(definition, syn, def_emb)
        scored.append((syn, s))

    scored.sort(key=lambda x: x[1], reverse=True)
    return genus, scored[:topk]

# -----------------------------
# 5) Valutazione: match termine (ITA) nei lemma del synset
# -----------------------------
def normalize_lemma_form(s: str) -> str:
    s = s.lower().strip()
    s = re.sub(r"\s+", " ", s)
    return s

def is_correct_prediction(term_it: str, syn: wn.synset) -> bool:
    t = normalize_lemma_form(term_it)
    lemmas_it = [normalize_lemma_form(l.replace("_", " ")) for l in syn.lemma_names(lang="ita")]
    return t in lemmas_it

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# -----------------------------
# 6) Run su tutto il dataset
# -----------------------------
rows_out = []
overall_total = 0
overall_correct = 0

for _, row in df.iterrows():
    categoria = str(row[cat_col]).strip()
    term = str(row[term_col]).strip()
    defs = get_definitions_for_row(row)
    if not defs:
        continue

    for idx, d in enumerate(defs, start=1):
        genus, top = best_synsets_for_definition(d, topk=5, depth=2)

        best_syn = top[0][0] if top else None
        best_score = top[0][1] if top else None

        correct = False
        if best_syn is not None:
            correct = is_correct_prediction(term, best_syn)
            overall_total += 1
            overall_correct += int(correct)

        rows_out.append({
            "Categoria": categoria,
            "Termine": term,
            "Def_ID": f"P{idx}",
            "Genus_estratto": genus,
            "Best_synset": best_syn.name() if best_syn else None,
            "Best_score": round(best_score, 4) if best_score is not None else None,
            "Best_gloss_EN": best_syn.definition() if best_syn else None,
            "Best_lemmi_IT": ", ".join(best_syn.lemma_names(lang="ita")) if best_syn else None,
            "Correct": correct
        })

        # stampa breve (facoltativo)
        print("="*80)
        print(f"[{categoria}] Termine: {term} | Def {idx}")
        print("Definizione:", d)
        print("Genus estratto:", genus)
        if not top:
            print("Nessun candidato WordNet trovato con genus -> prova depth maggiore o genus diverso.")
            continue
        print("\nTop candidati:")
        for syn, sc in top:
            ok = "✅" if is_correct_prediction(term, syn) else ""
            print(f"  {syn.name():25s} score={sc:.4f} {ok}")
            # stampa lemma IT principali
            lem_it = syn.lemma_names(lang="ita")
            if lem_it:
                print(f"    lemmi_IT: {', '.join(lem_it[:6])}")
            print(f"    gloss_EN: {syn.definition()}")

# -----------------------------
# 7) Report finale
# -----------------------------
res = pd.DataFrame(rows_out)

print("\n\n=== RISULTATI (prime righe) ===")
print(res.head(10))

if overall_total > 0:
    acc = overall_correct / overall_total
    print(f"\n=== Accuracy (best synset contains termine IT) ===")
    print(f"{overall_correct}/{overall_total} = {acc:.2%}")
else:
    print("\nNessuna predizione valutabile (controlla dataset o candidati).")

[CG] Termine: Pantalone | Def 1
Definizione: Indumento per la parte inferiore del corpo umano
Genus estratto: indumento

Top candidati:
  singlet.n.01              score=0.6780 
    lemmi_IT: camiciola, maglia, maglia_intima, maglietta
    gloss_EN: a collarless men's undergarment for the upper part of the body
  underpants.n.01           score=0.5951 
    lemmi_IT: mutande
    gloss_EN: an undergarment that covers the body from the waist no further than to the thighs; usually worn next to the skin
  foundation_garment.n.01   score=0.5409 
    lemmi_IT: busto, guaina
    gloss_EN: a woman's undergarment worn to give shape to the contours of the body
  shirt.n.01                score=0.5399 
    lemmi_IT: camicia
    gloss_EN: a garment worn on the upper half of the body
  underwear.n.01            score=0.5134 
    lemmi_IT: biancheria, biancheria_intima, intimo
    gloss_EN: undergarment worn next to the skin and under the outer garments
[CG] Termine: Pantalone | Def 2
Definizione: In

In [ ]:
# -----------------------------
# 8) TOP5 assoluta per termine
# -----------------------------
rows_out = []
overall_total = 0
overall_correct = 0

print("Elaborazione in corso...\n")

for _, row in df.iterrows():
    categoria = str(row[cat_col]).strip()
    term = str(row[term_col]).strip()
    defs = get_definitions_for_row(row)

    if not defs:
        continue

    # dizionario per aggregare i risultati di tutte le definizioni del termine corrente.
    # chiave: nome del synset. Valore: (score, synset_obj, def_index, definition_text)
    # logica: "Max Pooling" -> se diverse definizioni propongono lo stesso synset, teniamo quello con lo score più alto.
    term_best_candidates = {}

    # 1. raccolta candidati da tutte le definizioni
    for idx, d in enumerate(defs, start=1):
        # aumento di topk per avere più materiale da aggregare
        genus, top = best_synsets_for_definition(d, topk=10, depth=2)

        for syn, score in top:
            s_name = syn.name()

            # se è la prima volta che vede questo synset / se ha trovato uno score migliore
            if s_name not in term_best_candidates or score > term_best_candidates[s_name]['score']:
                term_best_candidates[s_name] = {
                    'synset': syn,
                    'score': score,
                    'def_id': f"P{idx}",
                    'def_text': d,
                    'genus': genus
                }

    # 2. Ordinamento globale per il termine
    # trasforma i valori del dizionario in una lista e ordina per score decrescente
    sorted_candidates = sorted(term_best_candidates.values(), key=lambda x: x['score'], reverse=True)

    # prende i top 5 assoluti
    top_5_global = sorted_candidates[:5]

    # 3. Calcolo metriche e salvataggio righe (basato sul migliore in assoluto)
    if top_5_global:
        best_candidate = top_5_global[0]
        best_syn = best_candidate['synset']
        is_correct = is_correct_prediction(term, best_syn)

        overall_total += 1
        overall_correct += int(is_correct)

        # salva nel report la riga relativa al vincitore assoluto
        rows_out.append({
            "Categoria": categoria,
            "Termine": term,
            "Best_Def_Source": best_candidate['def_id'],
            "Genus_estratto": best_candidate['genus'],
            "Best_synset": best_syn.name(),
            "Best_score": round(best_candidate['score'], 4),
            "Best_gloss_EN": best_syn.definition(),
            "Correct": is_correct
        })

    # 4. Stampa Output Formattato (Richiesta Utente)
    print("="*80)
    print(f"TERMINE: {term.upper()} ({categoria})")
    print(f"Definizioni analizzate: {len(defs)}")

    if not top_5_global:
        print("  -> Nessun candidato trovato.")
    else:
        print(f"\n  TOP 5 RISULTATI SIGNIFICATIVI:")
        for rank, cand in enumerate(top_5_global, 1):
            syn = cand['synset']
            sc = cand['score']
            origin = cand['def_id']

            # check visuale se è corretto
            match_icon = "✅" if is_correct_prediction(term, syn) else "❌"

            print(f"  {rank}. {syn.name():<25} | Score: {sc:.4f} | {match_icon} | (da {origin})")
            print(f"     Glossa EN: {syn.definition()}")
            lemmi_it = syn.lemma_names(lang="ita")
            if lemmi_it:
                print(f"     Lemmi IT : {', '.join(lemmi_it[:5])}")
            print("-" * 40)
    print("\n")

Elaborazione in corso...

TERMINE: PANTALONE (CG)
Definizioni analizzate: 40

  🏆 TOP 5 RISULTATI SIGNIFICATIVI:
  1. footwear.n.01             | Score: 0.8372 | ❌ | (da P3)
     Glossa EN: clothing worn on a person's feet
     Lemmi IT : calzatura
----------------------------------------
  2. legging.n.01              | Score: 0.8337 | ❌ | (da P12)
     Glossa EN: a garment covering the leg (usually extending from the knee to the ankle)
     Lemmi IT : gambale
----------------------------------------
  3. tailor-made.n.01          | Score: 0.8303 | ❌ | (da P35)
     Glossa EN: custom-made clothing
----------------------------------------
  4. ready-to-wear.n.01        | Score: 0.8128 | ❌ | (da P35)
     Glossa EN: ready-made clothing
----------------------------------------
  5. shirt.n.01                | Score: 0.8102 | ❌ | (da P33)
     Glossa EN: a garment worn on the upper half of the body
     Lemmi IT : camicia
----------------------------------------


TERMINE: MICROSCOPIO (CS